In [2]:
# TASK 1: Goals conceded By European vs Non-European teams in FIFA World Cup 2026
# Note: Comparision is done for the team who qualified for Round of 32 i.e. not considering teams eliminated in Group Stage
# Suresh Bhandari (S400969)

import pandas as pd
import numpy as np
import scipy.stats as st

In [ ]:
# information about the dataset used
df_all_record = pd.read_csv('dataset/goals_conceded_dataset.csv') 
print("=" * 50)
print(f"Total Number of Teams: {len(df_all_record)} teams")
print("=" * 50)
print(df_all_record[["Stage", "StageOrder"]].drop_duplicates().sort_values("StageOrder"))
print()
print(df_all_record["Stage"].value_counts())
print(df_all_record["Region"].value_counts())

Total Number of Teams: 48 teams
           Stage  StageOrder
32   Group Stage           1
16   Round of 32           2
8    Round of 16           3
4   Quarterfinal           4
3   Fourth Place           5
2    Third Place           6
1      Runner-up           7
0         Winner           8

Stage
Round of 32     16
Group Stage     16
Round of 16      8
Quarterfinal     4
Winner           1
Runner-up        1
Third Place      1
Fourth Place     1
Name: count, dtype: int64
Region
Non-Europe    32
Europe        16
Name: count, dtype: int64


In [4]:
# applying the filter to keep only the team that reached atleast Round of 32 i.e. Stage Order >= 2.
df_r32 = df_all_record[df_all_record["StageOrder"] >= 2].copy()
 
print("\n" + "=" * 75)
print(f"Final Record: {len(df_r32)} teams ({len(df_all_record) - len(df_r32)} "
      f"teams are eliminated in the group stage)")
print("=" * 75)
print(df_r32["Region"].value_counts())


Final Record: 32 teams (16 teams are eliminated in the group stage)
Region
Non-Europe    19
Europe        13
Name: count, dtype: int64


In [5]:
# assigning data to variables
europe = df_r32[df_r32["Region"] == "Europe"]["GoalsConcededPerMatch"].to_numpy()
non_europe = df_r32[df_r32["Region"] == "Non-Europe"]["GoalsConcededPerMatch"].to_numpy()
n1, n2 = len(europe), len(non_europe)

In [7]:
# descriptive statistics

print("\n" + "=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
for label, sample in [("European teams", europe), ("Non-European teams", non_europe)]:
    mean, median = np.mean(sample), np.median(sample)
    sd = np.std(sample, ddof=1)
    se = sd / np.sqrt(len(sample))
    print(f"\n{label} (n={len(sample)}):")
    print(f"  Mean={mean:.3f}  Median={median:.3f}  SD={sd:.3f}  SE={se:.3f}")
    print(f"  Min={sample.min():.2f}  Max={sample.max():.2f}")


DESCRIPTIVE STATISTICS

European teams (n=13):
  Mean=1.438  Median=1.300  SD=0.661  SE=0.183
  Min=0.10  Max=2.50

Non-European teams (n=19):
  Mean=1.163  Median=1.000  SD=0.509  SE=0.117
  Min=0.20  Max=2.30


In [9]:
# 95% confidence intervals (t-distribution, n<30)
print("\n" + "=" * 50)
print("95% CONFIDENCE INTERVALS")
print("=" * 50)
for label, sample in [("European teams", europe), ("Non-European teams", non_europe)]:
    n = len(sample)
    mean = np.mean(sample)
    se = np.std(sample, ddof=1) / np.sqrt(n)
    t_crit = st.t.ppf(0.975, df=n - 1)
    moe = t_crit * se
    print(f"{label}: mean={mean:.3f}, 95% CI = ({mean - moe:.3f}, {mean + moe:.3f})")


95% CONFIDENCE INTERVALS
European teams: mean=1.438, 95% CI = (1.039, 1.838)
Non-European teams: mean=1.163, 95% CI = (0.918, 1.409)


In [10]:
# two-sample t-test (Welch's, unequal variances)
print("\n" + "=" * 70)
print("TWO-SAMPLE T-TEST: Europe vs Non-Europe")
print("H0: mu_Europe = mu_NonEurope   Ha: mu_Europe != mu_NonEurope")
print("=" * 70)
t_stat, p_val = st.ttest_ind(europe, non_europe, equal_var=False)
print(f"t* = {t_stat:.3f}, p-value = {p_val:.4f}")
print("Reject H0" if p_val < 0.05 else "Fail to reject H0", "at alpha = 0.05")


TWO-SAMPLE T-TEST: Europe vs Non-Europe
H0: mu_Europe = mu_NonEurope   Ha: mu_Europe != mu_NonEurope
t* = 1.266, p-value = 0.2192
Fail to reject H0 at alpha = 0.05
